<a href="https://colab.research.google.com/github/tardigrade-dot/colab-script/blob/main/train/t2s_plus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install datasets transformers torch wandb

In [ ]:
from huggingface_hub import login

login(token="hf_token")

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hf_token')
print(f'hf_token:{hf_token}')
wandb_token = userdata.get('wandb')
print(f'wandb_token:{wandb_token}')

In [ ]:
著-著着
畫-画划
覆-覆复
鍊-炼链
乾-乾干
帳-帐账
藉-借藉

In [ ]:
import json
import re
from datasets import load_dataset
from collections import defaultdict


def load_ambiguous_chars(filepath: str) -> dict:
    ambiguous = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("-")
            if len(parts) != 2:
                continue
            key_char = parts[0].strip()
            candidates = list(parts[1].strip())
            if key_char and candidates:
                ambiguous[key_char] = candidates
    return ambiguous


def split_sentences(text: str) -> list:
    return [s.strip() for s in re.split(r'[。！？\n]', text) if len(s.strip()) > 5]


def build_samples(sentence: str, candidates_map: dict, window: int = 64) -> list:
    """
    candidates_map: key 是候选字（简体），value 是候选列表
    如 {'干': ['乾', '干'], '乾': ['乾', '干']}
    """
    samples = []
    for i, char in enumerate(sentence):
        if char not in candidates_map:
            continue
        start = max(0, i - window)
        end = min(len(sentence), i + window + 1)
        window_text = sentence[start:end]
        local_i = i - start
        masked = window_text[:local_i] + "[MASK]" + window_text[local_i + 1:]
        samples.append({
            "input": masked,
            "label": char,
            "candidates": "".join(candidates_map[char])
        })
    return samples


def build_candidates_map(ambiguous: dict) -> dict:
    """
    将 ambiguous 转换为以候选字为 key 的映射
    {'乾': ['乾', '干']} → {'乾': ['乾', '干'], '干': ['乾', '干']}
    """
    candidates_map = {}
    for candidates in ambiguous.values():
        for char in candidates:
            candidates_map[char] = candidates
    return candidates_map


def preprocess_and_save(ambiguous: dict, output_path: str):
    """
    阶段一：流式扫描原始数据集，过滤出包含歧义字的句子，保存到本地
    只需跑一次
    """
    # 从 values 提取所有候选字（简体）作为扫描集合
    ambiguous_chars_set = set(
        char for candidates in ambiguous.values() for char in candidates
    )
    print(f"扫描字符集共 {len(ambiguous_chars_set)} 个字")

    saved = 0
    sources = [
        ("TigerResearch/pretrain_zh", "train", "content"),
        ("shaowenchen/wiki_zh",       "train", "text"),
    ]

    with open(output_path, "w", encoding="utf-8") as f:
        for dataset_name, split, field in sources:
            print(f"\n处理 {dataset_name} ...")
            ds = load_dataset(dataset_name, split=split, streaming=True)
            scanned = 0

            for item in ds:
                text = item.get(field, "")
                if not text or not any(c in ambiguous_chars_set for c in text):
                    scanned += 1
                    continue

                for sent in split_sentences(text):
                    if any(c in ambiguous_chars_set for c in sent):
                        f.write(json.dumps({"text": sent}, ensure_ascii=False) + "\n")
                        saved += 1

                scanned += 1
                if scanned % 10000 == 0:
                    print(f"  已扫描 {scanned} 条，已保存 {saved} 句")

    print(f"\n完成，共保存 {saved} 句到 {output_path}")


def print_stats(counter: dict, ambiguous: dict):
    seen = set()
    groups = {}
    for candidates in ambiguous.values():
        group_key = "".join(sorted(candidates))
        if group_key not in seen:
            seen.add(group_key)
            groups[group_key] = candidates

    total = sum(counter.values())
    print(f"\n{'候选组':<10} {'字':<5} {'数量':>8}  {'占比':>6}")
    print("-" * 40)
    for group_key, candidates in sorted(groups.items()):
        for char in candidates:
            count = counter.get(char, 0)
            ratio = count / total * 100 if total > 0 else 0
            print(f"{group_key:<10} {char:<5} {count:>8}  {ratio:>5.1f}%")
        print()
    print(f"总计: {total} 条")


def load_and_sample(input_path: str, ambiguous: dict,
                    min_count: int = 50,
                    max_count: int = 10000) -> list:
    """
    阶段二：从本地文件读取，统计 + 过滤 + 采样
    可以反复调整参数跑
    """
    # 构建以候选字为 key 的映射
    candidates_map = build_candidates_map(ambiguous)
    ambiguous_chars_set = set(candidates_map.keys())

    # 统计各候选字在文件中的数量
    print("统计各候选字数量...")
    counter = defaultdict(int)
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            sent = json.loads(line)["text"]
            for char in sent:
                if char in ambiguous_chars_set:
                    counter[char] += 1

    print_stats(counter, ambiguous)

    # 过滤无效字对（任一候选字数量不足 min_count 则剔除整组）
    print("\n过滤无效字对...")
    seen = set()
    valid_candidates_map = {}
    for candidates in ambiguous.values():
        group_key = "".join(sorted(candidates))
        if group_key in seen:
            continue
        seen.add(group_key)
        counts = [counter.get(c, 0) for c in candidates]
        if min(counts) >= min_count:
            for char in candidates:
                valid_candidates_map[char] = candidates
        else:
            print(f"  剔除 {group_key}: {dict(zip(candidates, counts))}")

    print(f"✅ 保留候选字: {sorted(valid_candidates_map.keys())}")

    # 采样：每个候选字最多 max_count 条
    print(f"\n开始采样，每字最多 {max_count} 条...")
    char_counter = defaultdict(int)
    all_samples = []

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            # 所有字都达到上限则停止
            if all(char_counter[c] >= max_count for c in valid_candidates_map):
                break
            sent = json.loads(line)["text"]
            for sample in build_samples(sent, valid_candidates_map):
                label = sample["label"]
                if char_counter[label] < max_count:
                    all_samples.append(sample)
                    char_counter[label] += 1

    print(f"\n最终样本数: {len(all_samples)}")
    print("各候选字数量:")
    for char, cnt in sorted(char_counter.items()):
        print(f"  {char}: {cnt}")

    return all_samples


# ── 主流程 ──────────────────────────────────────────────────
# AMBIGUOUS = load_ambiguous_chars("ambiguous_chars.txt")
# print(f"加载歧义字: {len(AMBIGUOUS)} 个")

# # 阶段一：只需要跑一次
# preprocess_and_save(AMBIGUOUS, "filtered_sentences.jsonl")

# # 阶段二：可以反复调整参数跑
# samples = load_and_sample(
#     input_path="filtered_sentences.jsonl",
#     ambiguous=AMBIGUOUS,
#     min_count=50,
#     max_count=10000
# )

In [ ]:
#清理数据, 从hf下载datasets,过滤出需要的保存在本地
from huggingface_hub import snapshot_download
import pandas as pd
import threading
import json
import os

def download_dataset(repo_id: str, local_dir: str):
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=local_dir
    )
    print(f"下载完成: {local_dir}")

def process_parquet_file(filepath: str, ambiguous_chars_set: set,
                         output_path: str, lock: threading.Lock,
                         saved_count: list):
    df = pd.read_parquet(filepath)
    field = "content" if "content" in df.columns else "text"

    lines = []
    for text in df[field].dropna():
        if not any(c in ambiguous_chars_set for c in text):
            continue
        for sent in split_sentences(text):
            if any(c in ambiguous_chars_set for c in sent):
                lines.append(json.dumps({"text": sent}, ensure_ascii=False) + "\n")

    if lines:
        with lock:
            with open(output_path, "a", encoding="utf-8") as f:
                f.writelines(lines)
            saved_count[0] += len(lines)
            print(f"  {os.path.basename(filepath)}: +{len(lines)} 句，总计 {saved_count[0]}")

def preprocess_local(local_dir: str, ambiguous: dict,
                     output_path: str, num_threads: int = 4):
    ambiguous_chars_set = set(
        char for candidates in ambiguous.values() for char in candidates
    )

    # 找到所有 parquet 文件
    parquet_files = []
    for root, dirs, files in os.walk(local_dir):
        for f in files:
            if f.endswith(".parquet"):
                parquet_files.append(os.path.join(root, f))

    print(f"共 {len(parquet_files)} 个分片文件，使用 {num_threads} 线程处理")
    open(output_path, "w").close()  # 清空输出文件

    lock = threading.Lock()
    saved_count = [0]

    # 按 num_threads 分批处理
    for i in range(0, len(parquet_files), num_threads):
        batch = parquet_files[i:i + num_threads]
        threads = [
            threading.Thread(
                target=process_parquet_file,
                args=(fp, ambiguous_chars_set, output_path, lock, saved_count)
            )
            for fp in batch
        ]
        for t in threads:
            t.start()
        for t in threads:
            t.join()

    print(f"\n完成，共保存 {saved_count[0]} 句")


# ── 主流程 ──────────────────────────────────────────────────
AMBIGUOUS = load_ambiguous_chars("ambiguous_chars.txt")

# 下载（只需一次）
download_dataset("TigerResearch/pretrain_zh", "./data/pretrain_zh")
download_dataset("shaowenchen/wiki_zh",       "./data/wiki_zh")

# 多线程处理本地文件
preprocess_local("./data/pretrain_zh", AMBIGUOUS, "filtered_sentences.jsonl", num_threads=2)
preprocess_local("./data/wiki_zh",     AMBIGUOUS, "filtered_sentences.jsonl", num_threads=2)


In [ ]:
#统计上一步过滤后的本地数据, 给训练的内容做参考
def count_filtered_sentences(input_path: str, ambiguous: dict) -> dict:
    """
    统计 filtered_sentences.jsonl 里每个候选字的数量
    按繁体 | 简体 | 数量 格式显示
    """
    # 构建候选字扫描集合
    ambiguous_chars_set = set(
        char for candidates in ambiguous.values() for char in candidates
    )

    # 统计每个字出现次数
    counter = defaultdict(int)
    total_lines = 0
    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            sent = json.loads(line)["text"]
            for char in sent:
                if char in ambiguous_chars_set:
                    counter[char] += 1
            total_lines += 1
            if total_lines % 100000 == 0:
                print(f"  已处理 {total_lines} 行...")

    # 按繁体 key 对应关系打印
    print(f"\n{'繁体':<6} | {'简体':<6} | {'数量':>8}")
    print("-" * 30)

    seen_groups = set()
    for trad_char, candidates in sorted(ambiguous.items()):
        group_key = "".join(sorted(candidates))
        if group_key in seen_groups:
            continue
        seen_groups.add(group_key)

        for simp_char in candidates:
            count = counter.get(simp_char, 0)
            print(f"{trad_char:<6} | {simp_char:<6} | {count:>8}")
        print()

    print(f"总行数: {total_lines}")
    print(f"总字数: {sum(counter.values())}")
    return dict(counter)


# 使用
AMBIGUOUS = load_ambiguous_chars("ambiguous_chars.txt")
counter = count_filtered_sentences("filtered_sentences.jsonl", AMBIGUOUS)


In [ ]:
#确认最终处理的字
def load_and_sample(input_path: str, ambiguous: dict,
                    min_count: int = 50,
                    max_count: int = 10000,
                    custom_max: dict = None) -> list:
    """
    custom_max: 对特定候选字单独设置上限
    如 {'借': 5000} 表示「借」最多收集 5000 条
    """
    candidates_map = build_candidates_map(ambiguous)
    ambiguous_chars_set = set(candidates_map.keys())

    # 统计数量
    print("统计各候选字数量...")
    counter = defaultdict(int)
    with open(input_path, "r", encoding="utf-8", buffering=1) as f:
        for line in f:
            for char in line:
                if char in ambiguous_chars_set:
                    counter[char] += 1

    print_stats(counter, ambiguous)

    # 过滤无效字对
    print("\n过滤无效字对...")
    seen = set()
    valid_candidates_map = {}
    for candidates in ambiguous.values():
        group_key = "".join(sorted(candidates))
        if group_key in seen:
            continue
        seen.add(group_key)
        counts = [counter.get(c, 0) for c in candidates]
        if min(counts) >= min_count:
            for char in candidates:
                valid_candidates_map[char] = candidates
        else:
            print(f"  剔除 {group_key}: {dict(zip(candidates, counts))}")

    print(f"✅ 保留候选字: {sorted(valid_candidates_map.keys())}")

    # 构建每个字的上限
    # 默认 max_count，custom_max 里的字单独覆盖
    char_max = {c: max_count for c in valid_candidates_map}
    if custom_max:
        for char, limit in custom_max.items():
            if char in char_max:
                char_max[char] = limit
                print(f"  自定义上限: {char} → {limit}")

    # 采样
    print(f"\n开始采样...")
    char_counter = defaultdict(int)
    all_samples = []

    with open(input_path, "r", encoding="utf-8", buffering=1) as f:
        for line in f:
            if all(char_counter[c] >= char_max[c] for c in valid_candidates_map):
                break
            sent = json.loads(line)["text"]
            for sample in build_samples(sent, valid_candidates_map):
                label = sample["label"]
                if char_counter[label] < char_max[label]:
                    all_samples.append(sample)
                    char_counter[label] += 1

    print(f"\n最终样本数: {len(all_samples)}")
    print("各候选字数量:")
    for char, cnt in sorted(char_counter.items()):
        limit = char_max[char]
        print(f"  {char}: {cnt} / {limit}")

    return all_samples


# ── 使用 ────────────────────────────────────────────────────
AMBIGUOUS = load_ambiguous_chars("ambiguous_chars.txt")

samples = load_and_sample(
    input_path="filtered_sentences.jsonl",
    ambiguous=AMBIGUOUS,
    min_count=50,
    max_count=10000,
    custom_max={"借": 5000}  # 借单独限制5000，与藉的4618接近
)



In [ ]:
#将数据集上传到hf
from datasets import Dataset
from huggingface_hub import login

def save_to_huggingface(samples: list, repo_id: str, token: str = None):
    """
    将样本保存为 HuggingFace 数据集并上传
    """
    if token:
        login(token=token)

    # 本地先保存一份 jsonl 备份
    local_path = "ambiguous_samples.jsonl"
    with open(local_path, "w", encoding="utf-8") as f:
        for sample in samples:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")
    print(f"本地备份已保存到 {local_path}，共 {len(samples)} 条")

    # 转成 HuggingFace Dataset 格式
    hf_dataset = Dataset.from_list(samples)
    print(f"\n数据集结构:")
    print(hf_dataset)

    # 上传
    hf_dataset.push_to_hub(
        repo_id,
        private=False  # True 则为私有数据集
    )
    print(f"\n✅ 已上传到 https://huggingface.co/datasets/{repo_id}")


# ── 使用 ────────────────────────────────────────────────────
save_to_huggingface(
    samples=samples,
    repo_id="tardigrade-doc/t2c-plus",
    token="hf_token"
)

In [ ]:
#模型训练
import torch
import json
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from torch.utils.data import Dataset as TorchDataset

# ── Step 1: 加载 tokenizer 和模型 ────────────────────────────
MODEL_NAME = "hfl/chinese-roberta-wwm-ext"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForMaskedLM.from_pretrained(MODEL_NAME)

# ── Step 2: 自定义 Dataset ───────────────────────────────────
class AmbiguousCharDataset(TorchDataset):
    def __init__(self, hf_dataset, tokenizer, max_length=128):
        self.data = hf_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        input_text  = item["input"]       # "随[MASK]情况的发展"
        label_char  = item["label"]       # "着"
        candidates  = list(item["candidates"])  # "著着" → ['著', '着']

        # tokenize，[MASK] 会被自动转成 mask token id
        encoding = self.tokenizer(
            input_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids      = encoding["input_ids"].squeeze()
        attention_mask = encoding["attention_mask"].squeeze()

        # 找到 [MASK] 的位置
        mask_pos = (input_ids == self.tokenizer.mask_token_id).nonzero(as_tuple=True)[0]

        # 构建 labels：只在 [MASK] 位置计算 loss，其余位置设为 -100
        labels = torch.full_like(input_ids, -100)
        if len(mask_pos) > 0:
            label_id = self.tokenizer.convert_tokens_to_ids(label_char)
            labels[mask_pos[0]] = label_id

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels
        }


# ── Step 3: 加载 HuggingFace 数据集 ─────────────────────────
print("加载数据集...")
raw_dataset = load_dataset("tardigrade-doc/t2c-plus", split="train")

# 切分训练集和验证集 90/10
split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = AmbiguousCharDataset(split["train"], tokenizer)
eval_dataset  = AmbiguousCharDataset(split["test"],  tokenizer)

print(f"训练集: {len(train_dataset)} 条")
print(f"验证集: {len(eval_dataset)} 条")

import numpy as np
import wandb

wandb.login(key=wandb_token)  # 从 https://wandb.ai/settings 获取

wandb.init(
    project="chinese-ambiguous-chars",   # 项目名
    name="roberta-wwm-ext-run1",         # 本次运行名称
    config={
        "model": "hfl/chinese-roberta-wwm-ext",
        "dataset": "tardigrade-doc/t2c-plus",
        "max_length": 64,
        "epochs": 3,
        "batch_size": 8,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-5,
    }
)

# ── Step 4: 训练参数 ─────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    logging_steps=100,
    report_to="wandb",        # ← 原来是 "none"
    run_name="roberta-wwm-ext-run1",  # wandb 运行名称
)


# ── Step 5: 自定义评估指标（候选字准确率）───────────────────


def preprocess_logits_for_metrics(logits, labels):
    """
    在 logits 存入内存之前，只保留 [MASK] 位置的预测结果
    shape: (batch, seq_len, vocab_size) → (batch, seq_len) 只保留 argmax
    """
    return logits.argmax(dim=-1)  # vocab_size 维度直接压缩掉


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # 此时 predictions shape = (N, seq_len)，不再是 (N, seq_len, vocab_size)

    mask_positions = labels != -100
    correct = 0
    total = 0

    for i in range(len(labels)):
        pos = mask_positions[i].nonzero()[0]
        if len(pos) == 0:
            continue
        p = pos[0]
        pred_id = predictions[i, p]
        true_id = labels[i, p]
        if pred_id == true_id:
            correct += 1
        total += 1

    return {"accuracy": correct / total if total > 0 else 0}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,  # ← 关键
)

print("开始训练...")
trainer.train()


# ── Step 7: 保存模型 ─────────────────────────────────────────
model.save_pretrained("./model_output")
tokenizer.save_pretrained("./model_output")
print("模型已保存到 ./model_output")


# ── Step 8: 上传到 HuggingFace ───────────────────────────────
from huggingface_hub import login

login(token=hf_token)
model.push_to_hub("tardigrade-doc/chinese-ambiguous-chars-model")
tokenizer.push_to_hub("tardigrade-doc/chinese-ambiguous-chars-model")
print("模型已上传到 HuggingFace")


In [ ]:
print(trainer.state.best_model_checkpoint)
# 应该输出 ./checkpoints/checkpoint-XXXX（第一个）

print(f"最优 eval/loss: {trainer.state.best_metric}")